# Model 2 — Fine-Tuned (LoRA, fp16)
**Team11 | WU LLM Course SS26**

Fine-tunes TinyLlama-1.1B on Austrian tax law Q&A pairs
generated from RIS statute texts (template-based, no API).

**Platform:** Kaggle free P100 GPU (16 GB VRAM)
**Method:** LoRA via peft + trl SFTTrainer — no bitsandbytes (P100 = CUDA sm_60, incompatible)
**Model:** TinyLlama/TinyLlama-1.1B-Chat-v1.0 (fp16, no quantization needed)
**Training data:** Template-generated Q&A from RIS statute sections

⚠ With ~150–200 training pairs the model learns **answer style and structure**,
not deep legal knowledge. This is expected — RAG (Model 3) will likely outperform it.

## Cell 1 — Install Libraries & Imports

Installs `transformers`, `peft`, `trl`, `datasets`, and related packages, then imports everything needed for training and inference. PyTorch is intentionally **not** reinstalled — the Kaggle pre-installed version supports the P100 GPU (CUDA sm_60), which newer pip builds drop.

**Impact:** All imports in one cell prevent `NameError` after a kernel restart. The PyTorch note is critical — a wrong install would silently break GPU training.

In [ ]:
# Cell 1 — Install libraries
# NOTE: do NOT install torch — Kaggle's pre-installed torch supports P100 (sm_60)
# reinstalling torch via pip pulls a version that drops sm_60 support and breaks
!pip install -q transformers accelerate peft trl datasets requests beautifulsoup4

# all imports here so kernel restarts don't break later cells
import os, re, time
import pandas as pd
import torch
import requests
from bs4 import BeautifulSoup
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 2 — Shared System Prompt

Defines the German-language system prompt used across all three models. This single constant tells the model to answer in German, stay concise (1–4 sentences), and cite the relevant Austrian statute.

**Impact:** Using the exact same prompt across Models 1, 2, and 3 ensures fair comparison — output differences reflect the model, not the instructions.

In [ ]:
# Cell 2 — Shared system prompt (same across all 3 models)
SYSTEM_PROMPT = (
    "Beantworte die folgende Frage zum österreichischen Steuerrecht auf Deutsch. "
    "Antworte in maximal 1–4 Sätzen. "
    "Nenne die einschlägige Rechtsnorm (z.B. § 7 Abs 1 KStG). "
    "Halluziniere keine Paragraphen. Wenn unklar, formuliere vorsichtig."
)

## Generate training data from RIS statute texts

We scrape Austrian law §§ from RIS and generate Q&A training pairs using templates.
No API needed — each answer is directly grounded in a real statute section.

## Cell 3 — Scrape Statute Text from RIS

Fetches the full text of five Austrian laws (EStG, KStG, UStG, GrEStG, BAO) from `ris.bka.gv.at` and splits each into individual § sections (up to 1 500 characters each).

**Impact:** Produces the raw legal text used to build training examples. The more statute sections scraped, the more diverse the training pairs — and better coverage means the fine-tuned model generalises to more question types.

In [ ]:
# Cell 3 — Scrape laws from RIS (same approach as Model 3)

LAWS = {
    "EStG 1988":   "10004570",
    "KStG 1988":   "10004569",
    "UStG 1994":   "10004873",
    "GrEStG 1987": "10004531",
    "BAO":         "10003940",
}

def scrape_law(name, gesetzesnummer):
    """Fetch full text of a law from RIS and split into § chunks."""
    url = f"https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer={gesetzesnummer}"
    print(f"  Fetching {name}...")
    resp = requests.get(url, timeout=60)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)

    # split on § boundaries
    parts = re.split(r'(?=§\s*\d+[a-z]?\.)', text)
    chunks = []
    for part in parts:
        part = part.strip()
        if not part or not part.startswith("§"):
            continue
        m = re.match(r'(§\s*\d+[a-z]?)\.?', part)
        section = m.group(1) if m else "?"
        if len(part) < 80:  # skip tiny fragments
            continue
        chunks.append({"law": name, "section": section, "text": part[:1500]})
    print(f"  → {name}: {len(chunks)} sections")
    return chunks

# scrape all laws
all_sections = []
for name, gnr in LAWS.items():
    try:
        all_sections.extend(scrape_law(name, gnr))
    except Exception as e:
        print(f"  ERROR: {name}: {e}")

print(f"\nTotal: {len(all_sections)} statute sections scraped.")

## Cell 4 — Generate Q&A Training Pairs

Creates template-based question–answer pairs from the scraped statute sections. Four question templates are rotated so the model sees varied phrasing. Every answer is grounded in real statute text — no external API is required.

**Impact:** Produces the training dataset (~750 pairs). The model learns the expected answer style and citation format from these examples. Because examples are template-based, the model learns style rather than deep legal knowledge.

In [ ]:
# Cell 4 — Generate Q&A training pairs from statute sections
# Template-based: each pair is grounded in an actual § text

# question templates — varied to teach the model different answer styles
TEMPLATES = [
    ("Was regelt {section} {law}?",
     "{section} {law} regelt Folgendes: {summary}"),
    ("Was besagt {section} {law}?",
     "Gemäß {section} {law}: {summary}"),
    ("Welche Regelung enthält {section} {law}?",
     "Die Regelung in {section} {law} besagt: {summary}"),
    ("Was ist der Inhalt von {section} {law}?",
     "Der Inhalt von {section} {law} umfasst: {summary}"),
]

def make_summary(text, max_len=300):
    """Extract the first meaningful sentences from a statute section."""
    # remove the § header line
    lines = text.split("\n")
    content = " ".join(line.strip() for line in lines[1:] if line.strip())
    # take first ~300 chars ending at a sentence boundary
    if len(content) > max_len:
        cut = content[:max_len].rfind(".")
        if cut > 100:
            content = content[:cut+1]
        else:
            content = content[:max_len] + "..."
    return content

# generate pairs
training_pairs = []
for sec in all_sections:
    summary = make_summary(sec["text"])
    if len(summary) < 30:  # skip sections with too little content
        continue

    # pick one template per section (rotate through templates)
    tmpl_q, tmpl_a = TEMPLATES[len(training_pairs) % len(TEMPLATES)]
    q = tmpl_q.format(section=sec["section"], law=sec["law"])
    a = tmpl_a.format(section=sec["section"], law=sec["law"], summary=summary)
    training_pairs.append({"question": q, "answer": a})

print(f"Generated {len(training_pairs)} training pairs.")
# show a few examples
for p in training_pairs[:3]:
    print(f"\nQ: {p['question']}")
    print(f"A: {p['answer'][:150]}...")

## Fine-tune with QLoRA

## Cell 5 — Load Base Model (fp16, No Quantization)

Downloads `TinyLlama/TinyLlama-1.1B-Chat-v1.0` and loads it in fp16 precision without 4-bit quantization. At 1.1B parameters the model fits in GPU memory without any compression.

**Impact:** Choosing TinyLlama avoids bitsandbytes compatibility issues (which requires CUDA sm_70+) while still running on a free T4. fp16 keeps training fast and memory-efficient.

In [ ]:
# Cell 5 — Load TinyLlama in fp16 (no bitsandbytes — P100 CUDA sm_60 incompatible)
# TinyLlama 1.1B fits easily in 16GB VRAM, trains fast on P100

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# load in fp16 — no quantization needed at 1.1B
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## Cell 6 — Apply LoRA Configuration

Wraps the base model with LoRA adapters targeting the `q_proj` and `v_proj` attention layers (rank 16, alpha 32). Only ~0.2 % of parameters become trainable.

**Impact:** LoRA makes fine-tuning feasible on a free GPU — instead of updating all 1.1B weights, only ~2 M adapter weights are trained, cutting memory and time dramatically while preserving the base model's general knowledge.

In [ ]:
# Cell 6 — Apply LoRA config (no prepare_model_for_kbit_training — not using 4-bit)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should be ~0.5% of total

## Cell 7 — Format Training Data as Chat Template

Converts each Q&A pair into a TinyLlama chat-template string (system prompt + user question + assistant answer) and wraps it in a HuggingFace `Dataset` object for SFTTrainer.

**Impact:** Correct formatting is critical — if the chat template is wrong, the model trains on malformed sequences and produces incoherent output at inference time.

In [ ]:
# Cell 7 — Format training data as instruction-tuning chat

def format_chat(pair):
    """Format a Q&A pair as a Mistral chat conversation."""
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + pair["question"]},
        {"role": "assistant", "content": pair["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# create HuggingFace dataset
train_data = Dataset.from_list(training_pairs)
train_data = train_data.map(format_chat)

print(f"Training dataset: {len(train_data)} examples")
print(f"\nExample formatted text (first 300 chars):")
print(train_data[0]["text"][:300])

## Cell 8 — Fine-Tune with SFTTrainer

Runs 3 epochs of LoRA fine-tuning using `SFTTrainer` with fp16, batch size 4, gradient accumulation 2, and learning rate 2e-4. The LoRA adapter weights are saved to `./lora_adapter` after training.

**Impact:** This is the core training cell. After it completes the model has been adapted to produce German legal answers in the expected citation style. Training 3 epochs on ~750 examples takes roughly 10 minutes on a T4 GPU.

In [ ]:
# Cell 8 — Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training done!")

model.save_pretrained("./lora_adapter")
print("LoRA adapter saved.")

## Inference on test set

## Cell 9 — Load Test Dataset

Searches standard Kaggle input paths for `dataset_clean.csv` and falls back to the working directory. Loads the 643 test questions into a DataFrame.

**Impact:** Loads the questions the fine-tuned model will answer. This file must **not** be used for training — it is the held-out test set evaluated against ground-truth answers.

In [ ]:
# Cell 9 — Load dataset_clean.csv from Kaggle input
# Upload dataset_clean.csv as a Kaggle dataset first (+ Add Data in sidebar)
import glob

kaggle_paths = glob.glob("/kaggle/input/**/dataset_clean.csv", recursive=True)
if kaggle_paths:
    csv_path = kaggle_paths[0]
elif os.path.exists("dataset_clean.csv"):
    csv_path = "dataset_clean.csv"
else:
    raise FileNotFoundError("Add dataset_clean.csv via '+ Add Data' in the Kaggle sidebar.")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} test questions from {csv_path}")

## Cell 10 — Run Inference on All 643 Questions

Iterates over every question and generates an answer with the fine-tuned model. Uses sampling (`temperature=0.5`, `top_p=0.9`, `repetition_penalty=1.3`, `max_new_tokens=150`). Gradient checkpointing is disabled beforehand to re-enable the KV cache for faster generation.

**Impact:** Produces all 643 answers. The repetition penalty prevents the model from looping, and the `<|user|>` EOS token stops generation before a second conversation turn begins.

In [ ]:
# Cell 10 — Inference auf allen 643 Fragen
results = []  # immer neu starten

# modell für inference vorbereiten
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

user_token_id = tokenizer.convert_tokens_to_ids("<|user|>")

for i, row in df.iterrows():
    try:
        messages = [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + row["prompt"]}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.5,
                top_p=0.9,
                repetition_penalty=1.3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=[tokenizer.eos_token_id, user_token_id],
            )

        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not answer:
            answer = "Keine Antwort generiert."
        results.append({"id": row["id"], "answer": answer})

    except Exception as e:
        print(f"Fehler bei {row['id']}: {e}")
        results.append({"id": row["id"], "answer": "Fehler bei der Generierung."})

    if len(results) % 50 == 0:
        print(f"  Progress: {len(results)}/{len(df)}")

print(f"Done. {len(results)} Antworten generiert.")


In [ ]:
print(f"results hat {len(results)} Einträge")
if len(results) > 0:
    print(results[0])

## Cell 11 — Save Results to CSV

Merges the generated answers back to the original question order and writes the output to `model2_finetuned.csv`. Uses a merge (instead of `.loc`) to handle any ID mismatches robustly.

**Impact:** Produces the final submission file. The merge guarantees row order matches the original dataset even if any questions were skipped during inference.

In [ ]:
# Cell 11 — Save results to CSV
output_path = "model2_finetuned.csv"
result_df = pd.DataFrame(results)
print(f"Results: {len(result_df)} Zeilen")

if len(result_df) == 0:
    print("ERROR: results ist leer — Cell 10 nochmal ausführen!")
else:
    # merge statt loc (robuster bei ID-Abweichungen)
    final_df = df[["id"]].merge(result_df, on="id", how="left")
    final_df.to_csv(output_path, index=False)
    print(f"Gespeichert: {output_path}")
    print("Download: File Browser links → Rechtsklick → Download")

## Cell 12 — Validate Submission Format

Checks the output CSV for the required format: correct column names, correct row count, matching IDs, no null answers, no blank answers, and no excessive answer repetition.

**Impact:** A final safety check before submission — catches formatting errors that would silently fail during automated evaluation and cost points.

In [ ]:
# Cell 12 — Validate submission format
ref = pd.read_csv(csv_path)
sub = pd.read_csv(output_path)

assert list(sub.columns) == ["id", "answer"], "Wrong columns"
assert len(sub) == len(ref), f"Expected {len(ref)} rows, got {len(sub)}"
assert sub["id"].tolist() == ref["id"].tolist(), "IDs don't match"
assert sub["id"].nunique() == len(sub), "Duplicate IDs"
assert sub["answer"].notna().all(), "Null answers found"
assert (sub["answer"].astype(str).str.strip() != "").all(), "Blank answers found"
top_freq = sub["answer"].value_counts().iloc[0]
assert top_freq < 20, f"Repeated answer {top_freq}x — possible crash"

print(f"OK — {output_path} is valid ({len(sub)} rows)")
print("\nSample answers:")
for _, row in sub.head(3).iterrows():
    print(f"\n{row['id']}: {row['answer'][:150]}...")